This notebook estimates domestic production domestically consumed for vehicles goods, based on the production data of Organisation Internationale des Constructeurs Automobiles (OICA) or International Organization of Motor Vehicle Manufacturers.

In [9]:
import pandas as pd
import sqlite3
import country_converter as coco

Connect to the SQL databases and extract relevant trade data for vehicles

In [76]:
conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/trade_data.db')

full_conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/baci_trade_data_full_version.db')

car_data = pd.read_sql("SELECT * FROM [International trade data] WHERE cmdCode LIKE '8703%'", full_conn)
bus_data = pd.read_sql("SELECT * FROM [International trade data] WHERE cmdCode LIKE '8702%'", full_conn)
lcv_data = pd.read_sql("SELECT * FROM [International trade data] WHERE cmdCode IN ('870421', '870431')", full_conn)
truck_data = pd.read_sql("SELECT * FROM [International trade data] WHERE cmdCode IN ('870410', '870422', '870423', '870432', '870490')", full_conn)

In [84]:
# get the different import values
import_car = car_data.set_index(['cmdCode', 'refYear', 'importer', 'exporter']).drop(
    ['value (1,000 USD)'], axis=1).groupby(['cmdCode', 'refYear', 'importer']).sum().reset_index()
import_bus = bus_data.set_index(['cmdCode', 'refYear', 'importer', 'exporter']).drop(
    ['value (1,000 USD)'], axis=1).groupby(['cmdCode', 'refYear', 'importer']).sum().reset_index()
import_lcv = lcv_data.set_index(['cmdCode', 'refYear', 'importer', 'exporter']).drop(
    ['value (1,000 USD)'], axis=1).groupby(['cmdCode', 'refYear', 'importer']).sum().reset_index()
import_truck = truck_data.set_index(['cmdCode', 'refYear', 'importer', 'exporter']).drop(
    ['value (1,000 USD)'], axis=1).groupby(['cmdCode', 'refYear', 'importer']).sum().reset_index()

In [87]:
# get the different export values
export_car = car_data.set_index(['cmdCode', 'refYear', 'exporter']).drop(
    ['value (1,000 USD)', 'importer'], axis=1).groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()
export_bus = bus_data.set_index(['cmdCode', 'refYear', 'exporter']).drop(
    ['value (1,000 USD)', 'importer'], axis=1).groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()
export_lcv = lcv_data.set_index(['cmdCode', 'refYear', 'exporter']).drop(
    ['value (1,000 USD)', 'importer'], axis=1).groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()
export_truck = truck_data.set_index(['cmdCode', 'refYear', 'exporter']).drop(
    ['value (1,000 USD)', 'importer'], axis=1).groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()

In [115]:
# calculate the different net export values
net_exports_car = (
    export_car.drop('cmdCode', axis=1).groupby(['refYear', 'exporter']).sum().reset_index().set_index(['refYear', 'exporter']) -
    import_car.drop('cmdCode', axis=1).rename(columns={'importer':'exporter'}).groupby(['refYear', 'exporter']).sum().reset_index().set_index(['refYear', 'exporter'])
)

net_exports_bus = (
    export_bus.drop('cmdCode', axis=1).groupby(['refYear', 'exporter']).sum().reset_index().set_index(['refYear', 'exporter']) -
    import_bus.drop('cmdCode', axis=1).rename(columns={'importer':'exporter'}).groupby(['refYear', 'exporter']).sum().reset_index().set_index(['refYear', 'exporter'])
)

net_exports_lcv = (
    export_lcv.drop('cmdCode', axis=1).groupby(['refYear', 'exporter']).sum().reset_index().set_index(['refYear', 'exporter']) -
    import_lcv.drop('cmdCode', axis=1).rename(columns={'importer':'exporter'}).groupby(['refYear', 'exporter']).sum().reset_index().set_index(['refYear', 'exporter'])
)

net_exports_truck = (
    export_truck.drop('cmdCode', axis=1).groupby(['refYear', 'exporter']).sum().reset_index().set_index(['refYear', 'exporter']) -
    import_truck.drop('cmdCode', axis=1).rename(columns={'importer':'exporter'}).groupby(['refYear', 'exporter']).sum().reset_index().set_index(['refYear', 'exporter'])
)

In [119]:
# only keep positive net export balances
net_exports_car = net_exports_car[net_exports_car > 0].dropna()
net_exports_bus = net_exports_bus[net_exports_bus > 0].dropna()
net_exports_lcv = net_exports_lcv[net_exports_lcv > 0].dropna()
net_exports_truck = net_exports_truck[net_exports_truck > 0].dropna()

In [246]:
# read the data from OICA
data = pd.read_excel('Data_formatted_2025_all.xlsx', None, index_col=0)

In [247]:
# only keep relevant years
data['Passenger cars'] = data['Passenger cars'].loc[:, [2021, 2022, 2023, 2024]]
data['Light commercial vehicles'] = data['Light commercial vehicles'].loc[:, [2021, 2022, 2023, 2024]]
data['Buses'] = data['Buses'].loc[:, [2021, 2022, 2023, 2024]]
data['Heavy duty trucks'] = data['Heavy duty trucks'].loc[:, [2021, 2022, 2023, 2024]]

In [248]:
# convert geos to ISO3 country codes
data['Passenger cars'].index = [coco.convert(i, to='ISO3') for i in data['Passenger cars'].index]
data['Light commercial vehicles'].index = [coco.convert(i, to='ISO3') for i in data['Light commercial vehicles'].index]
data['Buses'].index = [coco.convert(i, to='ISO3') for i in data['Buses'].index]
data['Heavy duty trucks'].index = [coco.convert(i, to='ISO3') for i in data['Heavy duty trucks'].index]

In [249]:
# We need weights for the vehicles to convert from "units" sold (in OICA) to tonnes (in BACI)
car_weight = 1314 # kg according to ecoinvent
LCV_weight = 3500 # kg according to Gemini3 (ecoinvent does not provide a weight)
bus_weight = 10000 # kg kinda random, too much range of possibilities
truck_weight = 7500 # kg kinda random, too much range of possibilities

In [250]:
# fill NA values with zeros, confidential values will be considered empty
data['Passenger cars'] = data['Passenger cars'].replace('Confidential', None)
data['Light commercial vehicles'] = data['Light commercial vehicles'].replace('Confidential', None)
data['Buses'] = data['Buses'].replace('Confidential', None)
data['Heavy duty trucks'] = data['Heavy duty trucks'].replace('Confidential', None)

In [251]:
# convert "units" of vehicles sold to tonnes - need to convert to match with BACI export data
data['Passenger cars'] *= (car_weight / 1000)
data['Light commercial vehicles'] *= (LCV_weight / 1000)
data['Buses'] *= (bus_weight / 1000)
data['Heavy duty trucks'] *= (truck_weight / 1000)

In [252]:
# remove net exports from total production of OICA
for year in [2021, 2022, 2023, 2024]:
    for country in data['Passenger cars'].index:
        if country in net_exports_car.loc[year].index:
            data['Passenger cars'].loc[country, year] -= net_exports_car.loc(axis=0)[year, country].iloc[0]
    for country in data['Light commercial vehicles'].index:
        if country in net_exports_lcv.loc[year].index:
            data['Light commercial vehicles'].loc[country, year] -= net_exports_lcv.loc(axis=0)[year, country].iloc[0]
    for country in data['Buses'].index:
        if country in net_exports_bus.loc[year].index:
            data['Buses'].loc[country, year] -= net_exports_bus.loc(axis=0)[year, country].iloc[0]
    for country in data['Heavy duty trucks'].index:
        if country in net_exports_truck.loc[year].index:
            data['Heavy duty trucks'].loc[country, year] -= net_exports_truck.loc(axis=0)[year, country].iloc[0]

In [253]:
# if it results in negative values -> assume the country is exporting 100% of its production
data['Passenger cars'] = data['Passenger cars'][data['Passenger cars'] > 0]
data['Light commercial vehicles'] = data['Light commercial vehicles'][data['Light commercial vehicles'] > 0]
data['Buses'] = data['Buses'][data['Buses'] > 0]
data['Heavy duty trucks'] = data['Heavy duty trucks'][data['Heavy duty trucks'] > 0]

In [254]:
# format data
data['Passenger cars'] = data['Passenger cars'].stack().reset_index()
data['Light commercial vehicles'] = data['Light commercial vehicles'].stack().reset_index()
data['Buses'] = data['Buses'].stack().reset_index()
data['Heavy duty trucks'] = data['Heavy duty trucks'].stack().reset_index()

data['Passenger cars'] = data['Passenger cars'].rename(columns={'level_0': 'producer', 'level_1': 'refYear', 0: 'quantity (t)'})
data['Light commercial vehicles'] = data['Light commercial vehicles'].rename(columns={'level_0': 'producer', 'level_1': 'refYear', 0: 'quantity (t)'})
data['Buses'] = data['Buses'].rename(columns={'level_0': 'producer', 'level_1': 'refYear', 0: 'quantity (t)'})
data['Heavy duty trucks'] = data['Heavy duty trucks'].rename(columns={'level_0': 'producer', 'level_1': 'refYear', 0: 'quantity (t)'})

In [255]:
# add commodity codes
data['Passenger cars'].loc[:, 'cmdCode'] = '870331'
df1 = data['Passenger cars'].copy()
df1.loc[:, 'cmdCode'] = '870380'
df2 = data['Passenger cars'].copy()
df2.loc[:, 'cmdCode'] = '870322'
# distribute equally between the three codes. It does not matter ebcause only the relative aspect matters, but still, let's do it.
data['Passenger cars'].loc[:, 'quantity (t)'] /= 3
df1.loc[:, 'quantity (t)'] /= 3
df2.loc[:, 'quantity (t)'] /= 3
data['Light commercial vehicles'].loc[:, 'cmdCode'] = '870421'
data['Buses'].loc[:, 'cmdCode'] = '8702'
data['Heavy duty trucks'].loc[:, 'cmdCode'] = '870422'
df3 = data['Heavy duty trucks'].copy()
df3.loc[:, 'cmdCode'] = '870423'
data['Heavy duty trucks'].loc[:, 'quantity (t)'] /= 2
df3.loc[:, 'quantity (t)'] /= 2

vehicles_production_data = pd.concat([
    data['Passenger cars'],
    df1,
    df2,
    data['Light commercial vehicles'],
    data['Buses'],
    data['Heavy duty trucks'],
    df3
])

In [256]:
vehicles_production_data.loc[:, 'source'] = 'Converted data from the OICA (International Organization of Motor Vehicle Manufacturers). Original data obtained from https://oica.net/production-statistics/.'
vehicles_production_data.loc[:, 'country_coverage'] = 'complete'

Add the data to the SQL database inside the "Real production" data table